# 已收市分钟尾部补齐
以NAS current为权威，实际会话分钟审计而非只看as_of日期。仅补SOXL/SOXX；保留旧坏行、同行日期及冻结研究版本。0训练、0回测。

In [1]:
import os,json,qlib
from pathlib import Path
os.environ['MLFLOW_ALLOW_FILE_STORE']='true'
root=Path('/Users/rziiiii/dev/workspace');card=root/'research/closed_minute_tail_repair_20261014'
qlib.init(provider_uri=str(root/'data/qlib/soxl_intraday_1min'),region='us',kernels=1)
from src.workflow.closed_minute_tail_repair import run
summary=run(root=root,card=card,plan=json.loads((card/'plan.json').read_text()))
summary

{'experiment': 'closed_minute_tail_repair_20261014',
 'plan_sha256': 'e3daf0cfe21a35a9e60cd9b43140d857c5a3c2baf86eeb2a8b047a2b66ce3714',
 'decision': 'closed_core_tail_repaired_context_not_extended',
 'objective_achieved': False,
 'new_fits': 0,
 'new_backtests': 0,
 'before_source_id': 'c803e1a39598f509f4e485ae41a0057840529d00fd4cfdb85212f9729cfdf522',
 'after_source_id': '149ef504c382386397944b1e7af203b3eace9d2c9e43e09d9164c9a3691b302c',
 'core_as_of': '2026-10-13',
 'context_as_of': '2026-10-09',
 'raw_rows_before': 1561051,
 'raw_rows_after': 1561680,
 'new_minutes': 629,
 'missing_before': 629,
 'missing_after': 0,
 'invalid_rows_after': 1,
 'prefix_raw_prices_unchanged': True,
 'overlap_tail_rows': 151,
 'overlap_tail_rows_changed': 2,
 'native_audits': [{'freq': '1min',
   'rows_per_symbol': {'SOXL': 390, 'SOXX': 390},
   'status': 'passed'},
  {'freq': '5min',
   'rows_per_symbol': {'SOXL': 78, 'SOXX': 78},
   'status': 'passed'},
  {'freq': '15min',
   'rows_per_symbol': {'SOX

# 已收市核心分钟尾部：补齐629分钟

0拟合、0回测。NAS登记as_of已是2026-10-13，但实际检查发现该日只存SOXL75分钟、SOXX76分钟，分别缺315/314分钟。日期标签与完整已收市会话不同，不能以as_of本身当作训练就绪证据。补采前本地原始缓存SHA与NAS权威对象一致，未以本地缓存替代NAS来源。

复用已有`refresh_published_core`与Futu只读历史接口，固定NONE/RTH，仅请求2026-10-13两只核心ETF；保留全部旧日期和源异常。OpenD与SDK已核验可用，没有改框架、交易配置或调度器。

|项目|补采前|补采后|
|---|---:|---:|
|原始总行数|1561051|1561680|
|缺失已到期分钟|629|0|
|SOXL 10月13日分钟数|75|390|
|SOXX 10月13日分钟数|76|390|

原生Qlib读回该日：每资产1分钟390行、5分钟78行、15分钟26行；价格非空、1分钟OHLC有效，聚合完整标志通过。10月13日之前的原始OHLC、volume、turnover逐值不变。

当天151条重叠旧记录中2条开盘价被源更新：SOXL/SOXX 09:31的绝对差分别0.01/0.18，其余比较字段未变。两条旧记录曾标记is_final=True，因此这个按时间推断的标志不等于“永不修订”或历史首发认证。旧NAS版本、旧接收时间及新差异均保留，不能用新值重写过去的前向记录。

## NAS与剩余边界

- 原版本：`c803e1a39598f509f4e485ae41a0057840529d00fd4cfdb85212f9729cfdf522`。
- 新版本：`149ef504c382386397944b1e7af203b3eace9d2c9e43e09d9164c9a3691b302c`，已写入NAS不可变快照并登记current，登记回读及本地原始SHA一致。
- 核心数据截至10月13日；同行5分钟上下文仍截至10月9日，未将继承对象冒充新采集。后续多源模型须遵守各自可用性，不能把全部输入宣称统一到13日。
- SOXX 2020-03-16的一条源OHLC异常仍保留为unknown/needs_data，未插值或删除压力日。10月13日SOXX另有一条VWAP范围提示，保留为成交额/成交量口径诊断，未改价量；完整性通过不等于所有质量问题消失。
- 新补采的历史值和接收时刻不是原始实时首收；旧实验的10月9日冻结源及研究结果未重写。回溯复权与真实分红现金流/券商成交资格也未由本次补采解决。

决定：**closed_core_tail_repaired_context_not_extended**。核心尾部缺口修复完成；没有新收益结论，稳定净SOXX超额目标未完成，生产release SHA与策略参数未改。

下一步先使必要的多源输入在相同已完成会话上可核验，再开展新冻结研究或前向观察；不能把补采后的重放叫作独立前向alpha。不要再次为相同闭市尾部无缺口场景重复补采。

[补采前](before_coverage.csv)、[补采后](after_coverage.csv)、[剩余异常](remaining_anomalies.csv)、[刷新回执](refresh_receipt.json)、[Recorder清单](verification_manifest.json)。4项既有分钟完整性/补采/NAS登记检查通过，Notebook执行、三频率原生读回及NAS研究证据回读通过。SDK日志位于默认仓库外目录；大原始数据与证据均由NAS持久保存。
